# Pokémon Conditional Diffusion Model

Generates new Pokémon sprites conditioned on **type** (18), **style** (3: 3d, sugimori, sprite), **evolution stage** (3: base, evo1, evo2), and an **optional previous evolution image**.

## Setup
- Dataset: 3 JSON files (`3d.json`, `sugimori.json`, `sprite.json`)
- Each entry: `{"target_image": ..., "prev_evo_image": ... or null, "type": ..., "stage": ..., "style": ...}`
- Images: RGBA, preprocessed to 64×64

In [ ]:
# ============================================================
# Cell 1: Imports & Config
# ============================================================
import os
import json
import math
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Cell 2: Hyperparameters
# ============================================================

# --- Paths ---
DATA_DIR = "./data"                # directory containing JSON files + images
JSON_FILES = ["3d.json", "sugimori.json", "sprite.json"]
OUTPUT_DIR = "./outputs"
CHECKPOINT_DIR = "./checkpoints"

# --- Image ---
IMG_SIZE = 64
IMG_CHANNELS = 4                   # RGBA

# --- Conditioning dimensions ---
NUM_TYPES = 18                     # official Pokémon types
NUM_STYLES = 3                     # 3d, sugimori, sprite
NUM_STAGES = 3                     # base, evo1, evo2
COND_EMBED_DIM = 128               # embedding size for the combined conditioning vector
PREV_EVO_CHANNELS = 4              # RGBA channels of the prev evo image (or zeros if none)

# --- Diffusion ---
TIMESTEPS = 1000
BETA_START = 1e-4
BETA_END = 0.02

# --- Training ---
BATCH_SIZE = 32
NUM_EPOCHS = 200
LEARNING_RATE = 2e-4
EMA_DECAY = 0.9999

# --- U-Net ---
BASE_CH = 128                      # base channel width
CH_MULTS = (1, 2, 2, 4)           # channel multipliers per resolution level
NUM_RES_BLOCKS = 2                 # residual blocks per level
ATTN_RESOLUTIONS = [16, 8]         # apply self-attention at these spatial resolutions
DROPOUT = 0.1

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# Cell 3: Attribute Mappings
# ============================================================

TYPE_TO_IDX = {
    "normal": 0, "fire": 1, "water": 2, "electric": 3,
    "grass": 4, "ice": 5, "fighting": 6, "poison": 7,
    "ground": 8, "flying": 9, "psychic": 10, "bug": 11,
    "rock": 12, "ghost": 13, "dragon": 14, "dark": 15,
    "steel": 16, "fairy": 17,
}

STYLE_TO_IDX = {
    "3d": 0, "sugimori": 1, "sprite": 2,
}

STAGE_TO_IDX = {
    "base": 0, "evo1": 1, "evo2": 2,
}

print(f"Types: {len(TYPE_TO_IDX)}, Styles: {len(STYLE_TO_IDX)}, Stages: 3")

In [ ]:
# ============================================================
# Cell 4: Dataset
# ============================================================

class PokemonDiffusionDataset(Dataset):
    """
    Loads datapoints from multiple JSON files.
    Each JSON entry:
    {
        "target_image": "path/to/target.png",
        "prev_evo_image": "path/to/prev.png" or null,
        "type": "grass" or ["grass", "poison"],
        "stage": "base" | "evo1" | "evo2" | 1 | 2 | 3,
        "style": "sprite" | "3d" | "sugimori"
    }
    """

    def __init__(self, data_dir, json_files, img_size=64):
        self.data_dir = Path(data_dir)
        self.img_size = img_size
        self.samples = []

        # Load and merge all JSON files
        for jf in json_files:
            path = self.data_dir / jf
            if not path.exists():
                print(f"Warning: {path} not found, skipping.")
                continue
            with open(path, "r") as f:
                entries = json.load(f)
            self.samples.extend(entries)
            print(f"Loaded {len(entries)} samples from {jf}")

        print(f"Total dataset size: {len(self.samples)}")

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),          # [0, 1]
            transforms.Lambda(lambda x: x * 2 - 1),  # [-1, 1]
        ])

    def _load_image(self, rel_path):
        """Load RGBA image and apply transforms."""
        full_path = self.data_dir / rel_path
        img = Image.open(full_path).convert("RGBA")
        return self.transform(img)

    def _encode_type(self, type_val):
        """
        Encode type as multi-hot vector (supports dual types).
        Input: "grass" or ["grass", "poison"]
        Output: tensor of shape (NUM_TYPES,)
        """
        vec = torch.zeros(NUM_TYPES)
        if isinstance(type_val, str):
            type_val = [type_val]
        for t in type_val:
            t_lower = t.lower().strip()
            if t_lower in TYPE_TO_IDX:
                vec[TYPE_TO_IDX[t_lower]] = 1.0
        return vec

    def _encode_style(self, style_val):
        """One-hot encode style."""
        vec = torch.zeros(NUM_STYLES)
        idx = STYLE_TO_IDX.get(style_val.lower().strip(), 0)
        vec[idx] = 1.0
        return vec

    def _encode_stage(self, stage_val):
        """One-hot encode evolution stage."""
        vec = torch.zeros(NUM_STAGES)
        idx = STAGE_TO_IDX.get(stage_val, 0)
        vec[idx] = 1.0
        return vec

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Target image (what the model learns to generate)
        target = self._load_image(sample["target_image"])

        # Previous evolution image (or zeros if base Pokémon)
        if sample.get("prev_evo_image") is not None:
            prev_evo = self._load_image(sample["prev_evo_image"])
            has_prev = torch.tensor(1.0)
        else:
            prev_evo = torch.zeros(IMG_CHANNELS, self.img_size, self.img_size)
            has_prev = torch.tensor(0.0)

        # Attribute conditioning
        type_vec = self._encode_type(sample["type"])
        style_vec = self._encode_style(sample["style"])
        stage_vec = self._encode_stage(sample["stage"])

        # Concatenate all attribute vectors: [18 + 3 + 3] = 24-dim
        cond_vec = torch.cat([type_vec, style_vec, stage_vec], dim=0)

        return {
            "target": target,           # (4, 64, 64)
            "prev_evo": prev_evo,       # (4, 64, 64)
            "has_prev": has_prev,        # scalar
            "cond_vec": cond_vec,        # (24,)
        }

In [ ]:
# ============================================================
# Cell 5: Noise Schedule
# ============================================================

class NoiseSchedule:
    """Linear beta schedule with precomputed diffusion constants."""

    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cpu"):
        self.timesteps = timesteps

        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alpha_cumprod_prev = F.pad(self.alpha_cumprod[:-1], (1, 0), value=1.0)

        # For adding noise (forward process)
        self.sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod)
        self.sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - self.alpha_cumprod)

        # For denoising (reverse process)
        self.sqrt_recip_alpha = torch.sqrt(1.0 / self.alphas)
        self.posterior_variance = (
            self.betas * (1.0 - self.alpha_cumprod_prev) / (1.0 - self.alpha_cumprod)
        )

    def add_noise(self, x_0, t, noise=None):
        """
        Forward diffusion: q(x_t | x_0) = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
        """
        if noise is None:
            noise = torch.randn_like(x_0)

        sqrt_ab = self.sqrt_alpha_cumprod[t].view(-1, 1, 1, 1)
        sqrt_1m_ab = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)

        return sqrt_ab * x_0 + sqrt_1m_ab * noise, noise

    @torch.no_grad()
    def sample_step(self, model, x_t, t, cond_vec, prev_evo, has_prev):
        """
        Single reverse step: p(x_{t-1} | x_t).
        """
        betas_t = self.betas[t].view(-1, 1, 1, 1)
        sqrt_1m_ab_t = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)
        sqrt_recip_a_t = self.sqrt_recip_alpha[t].view(-1, 1, 1, 1)

        # Predict noise
        t_tensor = torch.full((x_t.shape[0],), t, device=x_t.device, dtype=torch.long)
        pred_noise = model(x_t, t_tensor, cond_vec, prev_evo, has_prev)

        # Compute mean
        mean = sqrt_recip_a_t * (x_t - betas_t / sqrt_1m_ab_t * pred_noise)

        if t == 0:
            return mean
        else:
            var = self.posterior_variance[t].view(-1, 1, 1, 1)
            noise = torch.randn_like(x_t)
            return mean + torch.sqrt(var) * noise

In [ ]:
# ============================================================
# Cell 6: U-Net Building Blocks
# ============================================================

class SinusoidalPosEmb(nn.Module):
    """Sinusoidal timestep embedding."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device, dtype=torch.float32) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)


class FiLM(nn.Module):
    """Feature-wise Linear Modulation: scale and shift feature maps."""
    def __init__(self, cond_dim, num_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, num_channels * 2),
        )

    def forward(self, x, cond):
        """x: (B, C, H, W), cond: (B, cond_dim) -> (B, C, H, W)"""
        gamma_beta = self.net(cond)  # (B, 2C)
        gamma, beta = gamma_beta.chunk(2, dim=1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)  # (B, C, 1, 1)
        beta = beta.unsqueeze(-1).unsqueeze(-1)
        return x * (1 + gamma) + beta


class ResBlock(nn.Module):
    """Residual block with FiLM conditioning and optional dropout."""
    def __init__(self, in_ch, out_ch, cond_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.film1 = FiLM(cond_dim, out_ch)

        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.dropout = nn.Dropout(dropout)

        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.film1(h, cond)
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.skip(x)


class SelfAttention(nn.Module):
    """Multi-head self-attention for spatial feature maps."""
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.attn = nn.MultiheadAttention(channels, num_heads, batch_first=True)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        h = h.view(B, C, H * W).transpose(1, 2)    # (B, HW, C)
        h, _ = self.attn(h, h, h)
        h = h.transpose(1, 2).view(B, C, H, W)
        return x + h

In [ ]:
# ============================================================
# Cell 7: U-Net Model
# ============================================================

class ConditionalUNet(nn.Module):
    """
    U-Net for conditional diffusion.

    Inputs:
      - x_t: noisy image (B, 4, 64, 64)
      - t: timestep (B,)
      - cond_vec: attribute vector (B, 24) — type(18) + style(3) + stage(3)
      - prev_evo: previous evolution image (B, 4, 64, 64) — zeros if none
      - has_prev: mask (B,) — 1.0 if prev exists, 0.0 otherwise
    """

    def __init__(
        self,
        img_channels=4,
        prev_channels=4,
        cond_vec_dim=24,        # 18 types + 3 styles + 3 stages
        cond_embed_dim=128,
        base_ch=128,
        ch_mults=(1, 2, 2, 4),
        num_res_blocks=2,
        attn_resolutions=(16, 8),
        dropout=0.1,
        img_size=64,
    ):
        super().__init__()
        self.img_size = img_size
        time_dim = cond_embed_dim
        total_cond_dim = cond_embed_dim  # unified conditioning dimension

        # --- Timestep embedding ---
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(cond_embed_dim),
            nn.Linear(cond_embed_dim, cond_embed_dim * 4),
            nn.SiLU(),
            nn.Linear(cond_embed_dim * 4, cond_embed_dim),
        )

        # --- Attribute conditioning embedding ---
        self.cond_mlp = nn.Sequential(
            nn.Linear(cond_vec_dim, cond_embed_dim),
            nn.SiLU(),
            nn.Linear(cond_embed_dim, cond_embed_dim),
        )

        # --- Previous evolution image encoder ---
        # Lightweight CNN to compress prev_evo into a vector
        self.prev_encoder = nn.Sequential(
            nn.Conv2d(prev_channels, 32, 4, 2, 1),    # 64 -> 32
            nn.SiLU(),
            nn.Conv2d(32, 64, 4, 2, 1),               # 32 -> 16
            nn.SiLU(),
            nn.Conv2d(64, 128, 4, 2, 1),              # 16 -> 8
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),                   # 8 -> 1
            nn.Flatten(),
            nn.Linear(128, cond_embed_dim),
        )

        # --- Combine all conditioning ---
        # time_emb + cond_emb + prev_emb -> unified
        self.cond_combine = nn.Sequential(
            nn.Linear(cond_embed_dim * 3, cond_embed_dim * 2),
            nn.SiLU(),
            nn.Linear(cond_embed_dim * 2, total_cond_dim),
        )

        # --- U-Net encoder ---
        in_ch = img_channels
        self.init_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        channels = [base_ch]
        ch = base_ch
        cur_res = img_size

        for level, mult in enumerate(ch_mults):
            out_ch = base_ch * mult
            for _ in range(num_res_blocks):
                layers = nn.ModuleList([ResBlock(ch, out_ch, total_cond_dim, dropout)])
                if cur_res in attn_resolutions:
                    layers.append(SelfAttention(out_ch))
                self.down_blocks.append(layers)
                ch = out_ch
                channels.append(ch)

            if level < len(ch_mults) - 1:
                self.down_samples.append(nn.Conv2d(ch, ch, 4, 2, 1))
                channels.append(ch)
                cur_res //= 2
            else:
                self.down_samples.append(nn.Identity())

        # --- Middle ---
        self.mid_block1 = ResBlock(ch, ch, total_cond_dim, dropout)
        self.mid_attn = SelfAttention(ch)
        self.mid_block2 = ResBlock(ch, ch, total_cond_dim, dropout)

        # --- U-Net decoder ---
        self.up_blocks = nn.ModuleList()
        self.up_samples = nn.ModuleList()

        for level, mult in reversed(list(enumerate(ch_mults))):
            out_ch = base_ch * mult

            for i in range(num_res_blocks + 1):
                skip_ch = channels.pop()
                layers = nn.ModuleList([
                    ResBlock(ch + skip_ch, out_ch, total_cond_dim, dropout)
                ])
                if cur_res in attn_resolutions:
                    layers.append(SelfAttention(out_ch))
                self.up_blocks.append(layers)
                ch = out_ch

            if level > 0:
                self.up_samples.append(
                    nn.ConvTranspose2d(ch, ch, 4, 2, 1)
                )
                cur_res *= 2
            else:
                self.up_samples.append(nn.Identity())

        # --- Output ---
        self.final_norm = nn.GroupNorm(8, ch)
        self.final_conv = nn.Conv2d(ch, img_channels, 3, padding=1)

    def forward(self, x, t, cond_vec, prev_evo, has_prev):
        """
        x: (B, 4, 64, 64)      noisy image
        t: (B,)                 timestep indices
        cond_vec: (B, 24)       attribute conditioning
        prev_evo: (B, 4, 64, 64) previous evolution image (or zeros)
        has_prev: (B,)          1.0 if prev exists, 0.0 otherwise
        """

        # Build conditioning
        t_emb = self.time_mlp(t)                                 # (B, 128)
        c_emb = self.cond_mlp(cond_vec)                          # (B, 128)
        p_emb = self.prev_encoder(prev_evo)                      # (B, 128)
        p_emb = p_emb * has_prev.unsqueeze(1)                    # zero out if no prev

        cond = self.cond_combine(
            torch.cat([t_emb, c_emb, p_emb], dim=1)
        )  # (B, 128)

        # Encoder
        h = self.init_conv(x)
        skips = [h]
        block_idx = 0

        for level in range(len(CH_MULTS)):
            for _ in range(NUM_RES_BLOCKS):
                layers = self.down_blocks[block_idx]
                h = layers[0](h, cond)          # ResBlock
                if len(layers) > 1:
                    h = layers[1](h)             # SelfAttention
                skips.append(h)
                block_idx += 1

            ds = self.down_samples[level]
            if not isinstance(ds, nn.Identity):
                h = ds(h)
                skips.append(h)

        # Middle
        h = self.mid_block1(h, cond)
        h = self.mid_attn(h)
        h = self.mid_block2(h, cond)

        # Decoder
        block_idx = 0
        for level in reversed(range(len(CH_MULTS))):
            for _ in range(NUM_RES_BLOCKS + 1):
                skip = skips.pop()
                h = torch.cat([h, skip], dim=1)
                layers = self.up_blocks[block_idx]
                h = layers[0](h, cond)
                if len(layers) > 1:
                    h = layers[1](h)
                block_idx += 1

            us = self.up_samples[level if level > 0 else 0]
            # index into up_samples by decode order
            us = self.up_samples[len(CH_MULTS) - 1 - level]
            if not isinstance(us, nn.Identity):
                h = us(h)

        h = self.final_conv(F.silu(self.final_norm(h)))
        return h


# Quick test
model = ConditionalUNet(
    img_channels=IMG_CHANNELS,
    prev_channels=PREV_EVO_CHANNELS,
    cond_vec_dim=NUM_TYPES + NUM_STYLES + NUM_STAGES,
    cond_embed_dim=COND_EMBED_DIM,
    base_ch=BASE_CH,
    ch_mults=CH_MULTS,
    num_res_blocks=NUM_RES_BLOCKS,
    attn_resolutions=ATTN_RESOLUTIONS,
    dropout=DROPOUT,
    img_size=IMG_SIZE,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

In [ ]:
# ============================================================
# Cell 8: EMA (Exponential Moving Average)
# ============================================================

class EMA:
    """Exponential Moving Average of model parameters for stable generation."""

    def __init__(self, model, decay=0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    @torch.no_grad()
    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(
                    param.data, alpha=1 - self.decay
                )

    def apply_shadow(self):
        """Replace model params with EMA params for inference."""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self):
        """Restore original params after inference."""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])
        self.backup = {}

In [ ]:
# ============================================================
# Cell 9: Training Loop
# ============================================================

def train():
    # Dataset & DataLoader
    dataset = PokemonDiffusionDataset(DATA_DIR, JSON_FILES, IMG_SIZE)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
    )

    # Model, optimizer, schedule
    model = ConditionalUNet(
        img_channels=IMG_CHANNELS,
        prev_channels=PREV_EVO_CHANNELS,
        cond_vec_dim=NUM_TYPES + NUM_STYLES + NUM_STAGES,
        cond_embed_dim=COND_EMBED_DIM,
        base_ch=BASE_CH,
        ch_mults=CH_MULTS,
        num_res_blocks=NUM_RES_BLOCKS,
        attn_resolutions=ATTN_RESOLUTIONS,
        dropout=DROPOUT,
        img_size=IMG_SIZE,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    noise_schedule = NoiseSchedule(TIMESTEPS, BETA_START, BETA_END, device)
    ema = EMA(model, decay=EMA_DECAY)

    print(f"Training on {len(dataset)} samples, {len(dataloader)} batches/epoch")
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
        for batch in pbar:
            target = batch["target"].to(device)         # (B, 4, 64, 64)
            prev_evo = batch["prev_evo"].to(device)     # (B, 4, 64, 64)
            has_prev = batch["has_prev"].to(device)     # (B,)
            cond_vec = batch["cond_vec"].to(device)     # (B, 24)

            # Sample random timesteps
            t = torch.randint(0, TIMESTEPS, (target.shape[0],), device=device)

            # Forward diffusion
            x_t, noise = noise_schedule.add_noise(target, t)

            # Predict noise
            pred_noise = model(x_t, t, cond_vec, prev_evo, has_prev)

            # MSE loss
            loss = F.mse_loss(pred_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

        # Save checkpoint every 25 epochs
        if (epoch + 1) % 25 == 0:
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "ema_shadow": ema.shadow,
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": avg_loss,
            }, f"{CHECKPOINT_DIR}/ckpt_epoch_{epoch+1}.pt")
            print(f"  Saved checkpoint at epoch {epoch+1}")

        # Generate samples every 25 epochs
        if (epoch + 1) % 25 == 0:
            generate_samples(model, ema, noise_schedule, epoch + 1)

    return model, ema


# Uncomment to train:
# model, ema = train()

In [ ]:
# ============================================================
# Cell 10: Sampling / Generation
# ============================================================

def make_cond_vec(type_names, style, stage):
    """
    Build a conditioning vector from human-readable attributes.
    type_names: str or list of str, e.g. "fire" or ["grass", "poison"]
    style: str, e.g. "sprite"
    stage: str or int, e.g. "base" or 1
    """
    type_vec = torch.zeros(NUM_TYPES)
    if isinstance(type_names, str):
        type_names = [type_names]
    for t in type_names:
        if t.lower() in TYPE_TO_IDX:
            type_vec[TYPE_TO_IDX[t.lower()]] = 1.0

    style_vec = torch.zeros(NUM_STYLES)
    style_vec[STYLE_TO_IDX.get(style.lower(), 0)] = 1.0

    stage_vec = torch.zeros(NUM_STAGES)
    stage_vec[STAGE_TO_IDX.get(stage, 0)] = 1.0

    return torch.cat([type_vec, style_vec, stage_vec])


@torch.no_grad()
def generate(
    model,
    noise_schedule,
    cond_vec,
    prev_evo_image=None,
    num_samples=1,
    guidance_scale=1.0,
):
    """
    Generate Pokémon images from noise.

    Args:
        model: trained ConditionalUNet
        noise_schedule: NoiseSchedule instance
        cond_vec: (24,) conditioning vector
        prev_evo_image: (4, 64, 64) tensor or None
        num_samples: how many images to generate
        guidance_scale: >1.0 for classifier-free guidance (if trained with dropout)
    """
    model.eval()

    # Expand conditioning for batch
    cond = cond_vec.unsqueeze(0).expand(num_samples, -1).to(device)

    if prev_evo_image is not None:
        prev = prev_evo_image.unsqueeze(0).expand(num_samples, -1, -1, -1).to(device)
        has_prev = torch.ones(num_samples, device=device)
    else:
        prev = torch.zeros(num_samples, IMG_CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
        has_prev = torch.zeros(num_samples, device=device)

    # Start from pure noise
    x = torch.randn(num_samples, IMG_CHANNELS, IMG_SIZE, IMG_SIZE, device=device)

    # Reverse diffusion
    for t in tqdm(reversed(range(noise_schedule.timesteps)), total=noise_schedule.timesteps, desc="Generating"):
        x = noise_schedule.sample_step(model, x, t, cond, prev, has_prev)

    # Clamp to [-1, 1] and convert to [0, 1]
    x = (x.clamp(-1, 1) + 1) / 2
    return x


def tensor_to_pil(tensor):
    """Convert (4, H, W) tensor in [0,1] to PIL RGBA image."""
    img = tensor.cpu().permute(1, 2, 0).numpy()
    img = (img * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(img, "RGBA")


def generate_samples(model, ema, noise_schedule, epoch):
    """Generate a grid of sample Pokémon for visual inspection."""
    ema.apply_shadow()

    test_configs = [
        {"type": "fire",              "style": "sprite",   "stage": "base"},
        {"type": ["grass", "poison"], "style": "sprite",   "stage": "evo1"},
        {"type": "water",             "style": "3d",       "stage": "base"},
        {"type": "electric",          "style": "sugimori", "stage": "base"},
    ]

    images = []
    for cfg in test_configs:
        cond = make_cond_vec(cfg["type"], cfg["style"], cfg["stage"])
        generated = generate(model, noise_schedule, cond, num_samples=2)
        for i in range(generated.shape[0]):
            images.append(tensor_to_pil(generated[i]))

    # Save grid
    grid_w = 4
    grid_h = 2
    grid_img = Image.new("RGBA", (grid_w * IMG_SIZE, grid_h * IMG_SIZE), (0, 0, 0, 0))
    for idx, img in enumerate(images[:grid_w * grid_h]):
        row, col = divmod(idx, grid_w)
        grid_img.paste(img, (col * IMG_SIZE, row * IMG_SIZE))

    grid_img.save(f"{OUTPUT_DIR}/samples_epoch_{epoch}.png")
    print(f"  Saved sample grid to {OUTPUT_DIR}/samples_epoch_{epoch}.png")

    ema.restore()

In [ ]:
# ============================================================
# Cell 11: Interactive Generation
# ============================================================

def generate_pokemon(
    model,
    ema,
    noise_schedule,
    type_names,
    style,
    stage,
    prev_evo_path=None,
    num_samples=4,
    save_path=None,
):
    """
    User-friendly generation function.

    Example:
        generate_pokemon(
            model, ema, noise_schedule,
            type_names=["fire", "flying"],
            style="sprite",
            stage="evo2",
            prev_evo_path="data/charizard_prev.png",
            num_samples=4,
        )
    """
    ema.apply_shadow()

    cond = make_cond_vec(type_names, style, stage)

    prev_evo = None
    if prev_evo_path is not None:
        transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x * 2 - 1),
        ])
        prev_img = Image.open(prev_evo_path).convert("RGBA")
        prev_evo = transform(prev_img)

    generated = generate(model, noise_schedule, cond, prev_evo, num_samples)

    results = []
    for i in range(generated.shape[0]):
        pil_img = tensor_to_pil(generated[i])
        results.append(pil_img)
        if save_path:
            base, ext = os.path.splitext(save_path)
            pil_img.save(f"{base}_{i}{ext}")

    ema.restore()
    return results


def generate_evolution_chain(
    model,
    ema,
    noise_schedule,
    type_names,
    style,
    num_stages=3,
):
    """
    Generate a full evolution chain: base -> evo1 -> evo2.
    Each stage uses the previous output as conditioning.
    """
    ema.apply_shadow()
    stages = ["base", "evo1", "evo2"][:num_stages]
    chain = []
    prev_tensor = None

    for stage in stages:
        cond = make_cond_vec(type_names, style, stage)
        generated = generate(model, noise_schedule, cond, prev_tensor, num_samples=1)

        pil_img = tensor_to_pil(generated[0])
        chain.append(pil_img)

        # Use this output as prev_evo for the next stage
        prev_tensor = generated[0]  # already in [0, 1], need to convert to [-1, 1]
        prev_tensor = prev_tensor * 2 - 1

    # Save chain as horizontal strip
    strip = Image.new("RGBA", (IMG_SIZE * len(chain), IMG_SIZE), (0, 0, 0, 0))
    for i, img in enumerate(chain):
        strip.paste(img, (i * IMG_SIZE, 0))
    strip.save(f"{OUTPUT_DIR}/evo_chain_{'_'.join(type_names) if isinstance(type_names, list) else type_names}_{style}.png")

    ema.restore()
    return chain


# Example usage (uncomment after training):
# results = generate_pokemon(
#     model, ema, noise_schedule,
#     type_names="fire",
#     style="sprite",
#     stage="base",
#     num_samples=4,
#     save_path="outputs/fire_sprite.png",
# )
#
# chain = generate_evolution_chain(
#     model, ema, noise_schedule,
#     type_names=["water"],
#     style="sprite",
#     num_stages=3,
# )

In [ ]:
# ============================================================
# Cell 12: Load Checkpoint & Resume
# ============================================================

def load_checkpoint(path, model=None, optimizer=None):
    """Load a training checkpoint."""
    ckpt = torch.load(path, map_location=device)

    if model is None:
        model = ConditionalUNet(
            img_channels=IMG_CHANNELS,
            prev_channels=PREV_EVO_CHANNELS,
            cond_vec_dim=NUM_TYPES + NUM_STYLES + NUM_STAGES,
            cond_embed_dim=COND_EMBED_DIM,
            base_ch=BASE_CH,
            ch_mults=CH_MULTS,
            num_res_blocks=NUM_RES_BLOCKS,
            attn_resolutions=ATTN_RESOLUTIONS,
            dropout=DROPOUT,
            img_size=IMG_SIZE,
        ).to(device)

    model.load_state_dict(ckpt["model_state_dict"])

    ema = EMA(model, decay=EMA_DECAY)
    if "ema_shadow" in ckpt:
        ema.shadow = ckpt["ema_shadow"]

    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    print(f"Loaded checkpoint from epoch {ckpt.get('epoch', '?')}, loss: {ckpt.get('loss', '?'):.4f}")
    return model, ema, ckpt.get("epoch", 0)


# Example:
# model, ema, start_epoch = load_checkpoint("checkpoints/ckpt_epoch_100.pt")
# noise_schedule = NoiseSchedule(TIMESTEPS, BETA_START, BETA_END, device)

In [ ]:
# ============================================================
# Cell 13: Dataset Validation Utility
# ============================================================

def validate_dataset(data_dir, json_files):
    """
    Validate dataset JSON files and check that all images exist.
    Prints a summary of issues found.
    """
    data_dir = Path(data_dir)
    issues = []
    stats = {
        "total": 0,
        "base_samples": 0,
        "evo_samples": 0,
        "by_style": {},
        "by_type": {},
        "by_stage": {},
    }

    for jf in json_files:
        path = data_dir / jf
        if not path.exists():
            issues.append(f"Missing JSON file: {path}")
            continue

        with open(path) as f:
            entries = json.load(f)

        for i, entry in enumerate(entries):
            stats["total"] += 1
            prefix = f"{jf}[{i}]"

            # Check required fields
            for field in ["target_image", "type", "stage", "style"]:
                if field not in entry:
                    issues.append(f"{prefix}: missing field '{field}'")

            # Check target image exists
            if "target_image" in entry:
                img_path = data_dir / entry["target_image"]
                if not img_path.exists():
                    issues.append(f"{prefix}: target image not found: {entry['target_image']}")

            # Check prev_evo image exists (if specified)
            prev = entry.get("prev_evo_image")
            if prev is not None:
                stats["evo_samples"] += 1
                prev_path = data_dir / prev
                if not prev_path.exists():
                    issues.append(f"{prefix}: prev_evo image not found: {prev}")
            else:
                stats["base_samples"] += 1

            # Validate type values
            type_val = entry.get("type", "")
            types = [type_val] if isinstance(type_val, str) else type_val
            for t in types:
                t_lower = t.lower().strip()
                if t_lower not in TYPE_TO_IDX:
                    issues.append(f"{prefix}: unknown type '{t}'")
                stats["by_type"][t_lower] = stats["by_type"].get(t_lower, 0) + 1

            # Track style
            style = entry.get("style", "").lower().strip()
            stats["by_style"][style] = stats["by_style"].get(style, 0) + 1

            # Track stage
            stage = entry.get("stage", "")
            stats["by_stage"][str(stage)] = stats["by_stage"].get(str(stage), 0) + 1

    # Print report
    print("=" * 50)
    print("DATASET VALIDATION REPORT")
    print("=" * 50)
    print(f"Total samples: {stats['total']}")
    print(f"  Base (no prev evo): {stats['base_samples']}")
    print(f"  Evolution (has prev): {stats['evo_samples']}")
    print(f"\nBy style: {stats['by_style']}")
    print(f"By stage: {stats['by_stage']}")
    print(f"By type:  {stats['by_type']}")

    if issues:
        print(f"\n⚠ {len(issues)} ISSUES FOUND:")
        for issue in issues[:20]:  # cap at 20
            print(f"  - {issue}")
        if len(issues) > 20:
            print(f"  ... and {len(issues) - 20} more")
    else:
        print("\n✓ No issues found!")

    return stats, issues


# Run validation:
# stats, issues = validate_dataset(DATA_DIR, JSON_FILES)